In [ ]:
# =================================================
# 1️⃣ Imports
# =================================================
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, GRU, BatchNormalization, Dropout, Dense, Concatenate, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import MinMaxScaler

# =================================================
# 2️⃣ Load Training Data
# =================================================
face_train_df = pd.read_excel('/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Facial Dataset/Train/ALL_Facial_Landmark_Train_Dataset.xls')
audio_train_df = pd.read_excel('/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Train/MFCC_train_audio.xls')

# =================================================
# 3️⃣ Align Training Sample Sizes
# =================================================
min_train_samples = min(len(face_train_df), len(audio_train_df))
face_train_df_trimmed = face_train_df.sample(n=min_train_samples, random_state=42).reset_index(drop=True)
audio_train_df_trimmed = audio_train_df.sample(n=min_train_samples, random_state=42).reset_index(drop=True)

y_train = face_train_df_trimmed['is_stroke_face'].to_numpy().astype(np.float32)
X_face = face_train_df_trimmed.drop(columns=['is_stroke_face'])
X_audio = audio_train_df_trimmed.drop(columns=['class'])

# Scaling
scaler_face = MinMaxScaler()
X_face_scaled = scaler_face.fit_transform(X_face)
scaler_audio = MinMaxScaler()
X_audio_scaled = scaler_audio.fit_transform(X_audio)

# Reshape for CNN-GRU
X_face_reshaped = X_face_scaled[:, :, np.newaxis]
X_audio_reshaped = X_audio_scaled[:, :, np.newaxis]

# =================================================
# 4️⃣ Build CNN-GRU Submodel
# =================================================
CONV_FILTERS = 32
GRU_UNITS = 64
L2_REG = 1e-4
DROPOUT_RATE = 0.3
MLP_UNITS_1 = 128
MLP_UNITS_2 = 64
LEARNING_RATE = 1e-4

def create_modality_submodel(input_shape, name):
    input_layer = Input(shape=input_shape, name=f'{name}_input')
    x = Conv1D(CONV_FILTERS, kernel_size=5, activation='relu', padding='same', kernel_regularizer=l2(L2_REG))(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(DROPOUT_RATE)(x)
    x = Conv1D(CONV_FILTERS*2, kernel_size=3, activation='relu', padding='same', kernel_regularizer=l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(DROPOUT_RATE)(x)
    x = GRU(GRU_UNITS, activation='tanh', return_sequences=False, kernel_regularizer=l2(L2_REG), recurrent_regularizer=l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT_RATE)(x)
    return Model(inputs=input_layer, outputs=x, name=f'{name}_model')

landmark_model = create_modality_submodel(X_face_reshaped.shape[1:], 'landmark')
mfcc_model = create_modality_submodel(X_audio_reshaped.shape[1:], 'mfcc')

# Fusion MLP
landmark_input = landmark_model.input
mfcc_input = mfcc_model.input
fused_features = Concatenate()([landmark_model.output, mfcc_model.output])
y = Dense(MLP_UNITS_1, activation='relu', kernel_regularizer=l2(L2_REG))(fused_features)
y = Dropout(DROPOUT_RATE)(y)
y = Dense(MLP_UNITS_2, activation='relu', kernel_regularizer=l2(L2_REG))(y)
y = Dropout(DROPOUT_RATE)(y)
output_layer = Dense(1, activation='sigmoid')(y)

model = Model(inputs=[landmark_input, mfcc_input], outputs=output_layer, name='cnn_gru_multimodal_fusion')
optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# =================================================
# 5️⃣ Stochastic Modality Masking Generator with Ratios
# =================================================
def stochastic_masking_generator(X1, X2, Y, batch_size):
    sample_indices = np.arange(X1.shape[0])
    while True:
        np.random.shuffle(sample_indices)
        X1_shuffled = X1[sample_indices]
        X2_shuffled = X2[sample_indices]
        Y_shuffled = Y[sample_indices]

        for i in range(0, X1.shape[0], batch_size):
            batch_indices = np.arange(i, min(i+batch_size, X1.shape[0]))
            batch_X1 = X1_shuffled[batch_indices].copy()
            batch_X2 = X2_shuffled[batch_indices].copy()
            batch_Y = Y_shuffled[batch_indices]

            dummy_X1 = np.zeros_like(batch_X1)
            dummy_X2 = np.zeros_like(batch_X2)

            mask_options = ['full', 'm1_only', 'm2_only']
            probabilities = [0.5, 0.25, 0.25]
            mask_states = np.random.choice(mask_options, size=len(batch_indices), p=probabilities)

            for j, state in enumerate(mask_states):
                if state == 'm1_only': batch_X2[j] = dummy_X2[j]
                elif state == 'm2_only': batch_X1[j] = dummy_X1[j]

            # Calculate masking ratios
            full_ratio = np.sum(mask_states == 'full') / len(batch_indices)
            m1_only_ratio = np.sum(mask_states == 'm1_only') / len(batch_indices)
            m2_only_ratio = np.sum(mask_states == 'm2_only') / len(batch_indices)

            yield ([batch_X1, batch_X2], batch_Y, (full_ratio, m1_only_ratio, m2_only_ratio))

# =================================================
# Full Stochastic Modality Masking Training Loop
# =================================================
BATCH_SIZE = 32
EPOCHS = 500

# Generator
train_generator = stochastic_masking_generator(X_face_reshaped, X_audio_reshaped, y_train, BATCH_SIZE)
STEPS_PER_EPOCH = X_face_reshaped.shape[0] // BATCH_SIZE + int(X_face_reshaped.shape[0] % BATCH_SIZE != 0)

# Metrics trackers
train_loss_metric = tf.keras.metrics.Mean(name="train_loss")
train_acc_metric = tf.keras.metrics.BinaryAccuracy(name="train_accuracy")

print("\nStarting training with stochastic modality masking...")

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss_metric.reset_state()
    train_acc_metric.reset_state()

    total_full_ratio = 0.0
    total_m1_only_ratio = 0.0
    total_m2_only_ratio = 0.0

    for step in range(STEPS_PER_EPOCH):
        # Get batch
        (X_batch_L, X_batch_M), y_batch, (full_ratio, m1_only_ratio, m2_only_ratio) = next(train_generator)

        # Make sure y_batch has shape (batch_size, 1)
        y_batch = y_batch[:, np.newaxis].astype(np.float32)

        with tf.GradientTape() as tape:
            y_pred = model([X_batch_L, X_batch_M], training=True)
            # Binary crossentropy loss
            loss = tf.keras.losses.binary_crossentropy(y_batch, y_pred)
            loss = tf.reduce_mean(loss) + sum(model.losses)  # include regularization

        # Compute gradients and apply
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        # Update metrics
        train_loss_metric.update_state(loss)
        train_acc_metric.update_state(y_batch, y_pred)

        # Accumulate masking ratios
        total_full_ratio += full_ratio
        total_m1_only_ratio += m1_only_ratio
        total_m2_only_ratio += m2_only_ratio

        # Optional progress print
        if (step + 1) % (STEPS_PER_EPOCH // 4 + 1) == 0:
            print(f"Step {step+1}/{STEPS_PER_EPOCH} - "
                  f"Loss: {train_loss_metric.result().numpy():.4f} - "
                  f"Acc: {train_acc_metric.result().numpy():.4f}", end='\r')

    # Epoch summary
    avg_loss = train_loss_metric.result().numpy()
    avg_acc = train_acc_metric.result().numpy()
    avg_full_ratio = total_full_ratio / STEPS_PER_EPOCH
    avg_m1_only_ratio = total_m1_only_ratio / STEPS_PER_EPOCH
    avg_m2_only_ratio = total_m2_only_ratio / STEPS_PER_EPOCH

    print(f"\nEpoch {epoch} summary -> Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.4f} | "
          f"Masking Ratios -> Full: {avg_full_ratio:.2f}, "
          f"Landmark Only: {avg_m1_only_ratio:.2f}, MFCC Only: {avg_m2_only_ratio:.2f}")

In [ ]:
# =================================================
# 7️⃣ Testing Loops
# =================================================
face_test_files = [
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Facial Dataset/Test/aug_BrightnessContrast_landmark.xls',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Facial Dataset/Test/aug_Gaussian_landmark.xls',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Facial Dataset/Test/aug_JPEG_landmark.xls',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Facial Dataset/Test/aug_MotionBlur_landmark.xls',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Facial Dataset/Test/aug_Shadow_landmark.xls'
]

audio_test_files = [
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Additive_White_Noise.csv',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Babble_Noise.csv',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Combined_Corruption.csv',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Each_Combined_Corruption.csv',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Highpass_Filtering.csv',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Lowpass_Filtering.csv',
    '/kaggle/input/stroke-ultimate-dataset/Dataset/Stroke Audio Dataset/Test/mfcc_Reverberation.csv'
]

# --- Landmark-only Test ---
print("\n=== Landmark-only Test ===")
for file in face_test_files:
    test_df = pd.read_excel(file)

    # Ensure columns match training data
    X_test_face = test_df[X_face.columns].copy()
    y_test_face = test_df['class'].to_numpy()[:, np.newaxis].astype(np.float32)

    # Scale and reshape
    X_test_face_scaled = scaler_face.transform(X_test_face.values)[:, :, np.newaxis]

    # Dummy audio input with correct shape
    dummy_audio = np.zeros((X_test_face_scaled.shape[0], X_audio_reshaped.shape[1], 1))

    # Evaluate
    loss, acc = model.evaluate([X_test_face_scaled, dummy_audio], y_test_face, verbose=0)
    print(f"{file.split('/')[-1]}: Accuracy = {acc:.4f}")

# --- MFCC-only Test ---
print("\n=== MFCC-only Test ===")
for file in audio_test_files:
    test_df = pd.read_csv(file)

    # Match column names exactly to training columns
    feature_cols = X_audio.columns.tolist()
    if len(feature_cols) == test_df.shape[1] - 1:
        # Assume last column is 'class'
        test_df.columns = feature_cols + ['class']
    else:
        raise ValueError(f"Column mismatch in {file}")

    X_test_audio = test_df[feature_cols].copy()
    y_test_audio = test_df['class'].to_numpy()[:, np.newaxis].astype(np.float32)

    # Scale and reshape
    X_test_audio_scaled = scaler_audio.transform(X_test_audio.values)[:, :, np.newaxis]

    # Dummy face input with correct shape
    dummy_face = np.zeros((X_test_audio_scaled.shape[0], X_face_reshaped.shape[1], 1))

    # Evaluate
    loss, acc = model.evaluate([dummy_face, X_test_audio_scaled], y_test_audio, verbose=0)
    print(f"{file.split('/')[-1]}: Accuracy = {acc:.4f}")